In [0]:
# Databricks notebook source

# MAGIC %md
# MAGIC # ✅ 07 · Validación End-to-End — Trust & Safety
# MAGIC
# MAGIC ## Objetivo
# MAGIC
# MAGIC Ejecutar un **Acceptance Gate** automático sobre el pipeline completo:
# MAGIC
# MAGIC ```text
# MAGIC Bronze
# MAGIC   ↓
# MAGIC DQ Bronze
# MAGIC   ↓
# MAGIC Silver
# MAGIC   ↓
# MAGIC DQ Silver
# MAGIC   ↓
# MAGIC Feature Engineering
# MAGIC   ↓
# MAGIC Trust & Safety Evaluation
# MAGIC   ↓
# MAGIC Risk Consolidation
# MAGIC   ↓
# MAGIC Gold
# MAGIC   ↓
# MAGIC Unity Catalog / Gobierno
# MAGIC   ↓
# MAGIC Power BI
# MAGIC ```
# MAGIC
# MAGIC ## Principios de validación
# MAGIC
# MAGIC 1. El `Evaluation Run` es la unidad de trazabilidad de Trust & Safety.
# MAGIC 2. Las reglas utilizadas por un Run se obtienen de la evidencia del propio Run,
# MAGIC    **no del catálogo ACTIVE actual**.
# MAGIC 3. Gold debe corresponder exactamente al `Evaluation Run` publicable.
# MAGIC 4. Los Quality Gates Bronze y Silver bloquean únicamente por reglas de
# MAGIC    criticidad **alta** con `paso = false`.
# MAGIC 5. La metadata de Unity Catalog forma parte del criterio de publicación.
# MAGIC
# MAGIC > Este notebook **no transforma datos**. Solo valida consistencia, trazabilidad,
# MAGIC > calidad y gobierno. Si una validación crítica falla, el Job termina `FAILED`.
# MAGIC
# MAGIC ## Orden esperado antes de ejecutar 07
# MAGIC
# MAGIC ```text
# MAGIC DQ Bronze
# MAGIC   ↓
# MAGIC Silver
# MAGIC   ↓
# MAGIC DQ Silver
# MAGIC   ↓
# MAGIC Features
# MAGIC   ↓
# MAGIC Trust & Safety
# MAGIC   ↓
# MAGIC Risk Consolidation
# MAGIC   ↓
# MAGIC Gold
# MAGIC   ↓
# MAGIC E6.3 Classification
# MAGIC   ↓
# MAGIC 07 Acceptance Gate
# MAGIC ```

# COMMAND ----------

# ============================================================
# 1. IMPORTS
# ============================================================

from pyspark.sql import functions as F


# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Configuración
# MAGIC
# MAGIC Se centralizan todos los activos que forman parte del Acceptance Gate.
# MAGIC
# MAGIC La validación de gobierno puede configurarse mediante:
# MAGIC
# MAGIC `STRICT_GOVERNANCE = True`
# MAGIC
# MAGIC Con este valor, la ausencia de clasificación de columnas sensibles
# MAGIC también bloquea la publicación.

# COMMAND ----------

# ============================================================
# 2. CONFIGURACIÓN
# ============================================================

CATALOG = "workspace"

BRONZE_SCHEMA = "yelp_bronze"
SILVER_SCHEMA = "yelp_silver"
GOLD_SCHEMA = "yelp_gold"
GOV_SCHEMA = "yelp_gov"

BRONZE = f"{CATALOG}.{BRONZE_SCHEMA}"
SILVER = f"{CATALOG}.{SILVER_SCHEMA}"
GOLD = f"{CATALOG}.{GOLD_SCHEMA}"
GOV = f"{CATALOG}.{GOV_SCHEMA}"


# ------------------------------------------------------------
# Bronze
# ------------------------------------------------------------

BRONZE_TABLES = [
    f"{BRONZE}.brz_business",
    f"{BRONZE}.brz_user",
    f"{BRONZE}.brz_review",
    f"{BRONZE}.brz_tip",
    f"{BRONZE}.brz_checkin"
]


# ------------------------------------------------------------
# Silver
# ------------------------------------------------------------

REVIEW_TABLE = f"{SILVER}.slv_review"
FEATURE_TABLE = f"{SILVER}.slv_review_features"
EVALUATION_TABLE = f"{SILVER}.slv_review_evaluation"
RISK_TABLE = f"{SILVER}.slv_review_risk"


# ------------------------------------------------------------
# Gobierno
# ------------------------------------------------------------

DQ_RULE_TABLE = f"{GOV}.dq_reglas"
DQ_RESULT_TABLE = f"{GOV}.dq_resultados"

RUN_TABLE = f"{GOV}.evaluation_run"
RULE_TABLE = f"{GOV}.evaluation_rule"
RULE_VERSION_TABLE = f"{GOV}.evaluation_rule_version"


# ------------------------------------------------------------
# Gold
# ------------------------------------------------------------

GOLD_TABLES = [
    "gld_dim_business",
    "gld_dim_user",
    "gld_dim_rule",
    "gld_dim_evaluation_run",
    "gld_fact_review_evaluation",
    "gld_review_risk",
    "gld_investigation_queue",
    "gld_kpi_trust_safety_overview",
    "gld_kpi_trust_safety_business",
    "gld_kpi_trust_safety_daily",
    "gld_kpi_trust_safety_rule"
]

GOLD_REVIEW_RISK = f"{GOLD}.gld_review_risk"
GOLD_FACT_EVALUATION = f"{GOLD}.gld_fact_review_evaluation"
GOLD_QUEUE = f"{GOLD}.gld_investigation_queue"
GOLD_OVERVIEW = f"{GOLD}.gld_kpi_trust_safety_overview"


# ------------------------------------------------------------
# Gobierno estricto
# ------------------------------------------------------------

STRICT_GOVERNANCE = True


print("=" * 78)
print("YELPCONNECT - ACCEPTANCE GATE END TO END")
print("=" * 78)
print(f"Catalog            : {CATALOG}")
print(f"Strict Governance  : {STRICT_GOVERNANCE}")
print("=" * 78)


# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Framework de validación
# MAGIC
# MAGIC Cada check registra:
# MAGIC
# MAGIC - etapa;
# MAGIC - validación;
# MAGIC - valor actual;
# MAGIC - valor esperado;
# MAGIC - severidad;
# MAGIC - resultado.
# MAGIC
# MAGIC ### Severidades
# MAGIC
# MAGIC | Severidad | Comportamiento |
# MAGIC |---|---|
# MAGIC | `CRITICAL` | Si falla, el Acceptance Gate termina `FAILED` |
# MAGIC | `WARNING` | Se reporta, pero no bloquea por sí sola |

# COMMAND ----------

# ============================================================
# 3. HELPERS
# ============================================================

checks = []


def register_check(
    stage,
    name,
    actual,
    expected,
    passed,
    severity="CRITICAL"
):

    passed = bool(passed)

    checks.append(
        (
            stage,
            name,
            str(actual),
            str(expected),
            severity,
            passed
        )
    )

    icon = "✔" if passed else ("⚠" if severity == "WARNING" else "❌")

    print(
        f"{icon} [{stage}] {name}: "
        f"{actual} "
        f"(esperado: {expected}) "
        f"[{severity}]"
    )

    return passed


def check_eq(
    stage,
    name,
    actual,
    expected,
    severity="CRITICAL"
):

    return register_check(
        stage,
        name,
        actual,
        expected,
        actual == expected,
        severity
    )


def check_true(
    stage,
    name,
    condition,
    expected="True",
    severity="CRITICAL"
):

    return register_check(
        stage,
        name,
        bool(condition),
        expected,
        bool(condition),
        severity
    )


def table_exists(
    table_name
):

    return spark.catalog.tableExists(
        table_name
    )


def require_table(
    stage,
    table_name
):

    exists = table_exists(
        table_name
    )

    check_true(
        stage,
        f"Existe {table_name}",
        exists
    )

    return exists


# COMMAND ----------

# MAGIC %md
# MAGIC # 4. Precondiciones
# MAGIC
# MAGIC Antes de validar contenido, se comprueba que existan los activos mínimos
# MAGIC requeridos por el pipeline.

# COMMAND ----------

# ============================================================
# 4. PRECONDICIONES
# ============================================================

required_core_tables = (
    BRONZE_TABLES
    + [
        REVIEW_TABLE,
        FEATURE_TABLE,
        EVALUATION_TABLE,
        RISK_TABLE,
        DQ_RULE_TABLE,
        DQ_RESULT_TABLE,
        RUN_TABLE,
        RULE_TABLE,
        RULE_VERSION_TABLE
    ]
    + [
        f"{GOLD}.{table_name}"
        for table_name in GOLD_TABLES
    ]
)


missing_tables = [
    table_name
    for table_name in required_core_tables
    if not table_exists(table_name)
]


check_eq(
    "PRECONDITION",
    "Tablas requeridas faltantes",
    len(missing_tables),
    0
)


if missing_tables:

    print()
    print("Tablas faltantes:")

    for table_name in missing_tables:
        print(f" - {table_name}")

    raise Exception(
        "❌ No se puede ejecutar el Acceptance Gate: "
        "faltan tablas requeridas."
    )


# COMMAND ----------

# MAGIC %md
# MAGIC # 5. Evaluation Run publicable
# MAGIC
# MAGIC Se selecciona el último Run que cumpla simultáneamente:
# MAGIC
# MAGIC - `execution_status = SUCCESS`;
# MAGIC - existe evidencia en `slv_review_risk`.
# MAGIC
# MAGIC Este `RUN_ID` se utiliza como ancla de trazabilidad para Evaluation,
# MAGIC Risk y Gold.

# COMMAND ----------

# ============================================================
# 5. IDENTIFICAR ÚLTIMO RUN PUBLICABLE
# ============================================================

risk_runs = (
    spark.table(
        RISK_TABLE
    )
    .select(
        "evaluation_run_id"
    )
    .distinct()
)


latest_run = (
    spark.table(
        RUN_TABLE
    )
    .alias("r")

    .filter(
        F.col(
            "r.execution_status"
        ) == "SUCCESS"
    )

    .join(
        risk_runs.alias("rr"),
        "evaluation_run_id",
        "inner"
    )

    .orderBy(
        F.desc(
            "r.started_at"
        )
    )

    .select(
        "evaluation_run_id",
        "pipeline_name",
        "pipeline_version",
        "dataset_version",
        "started_at",
        "finished_at",
        "execution_status"
    )

    .first()
)


if latest_run is None:

    raise Exception(
        """
        ❌ No existe Evaluation Run SUCCESS
        con Risk Consolidation.
        """
    )


RUN_ID = latest_run.evaluation_run_id
DATASET_VERSION = latest_run.dataset_version


print()
print(f"Evaluation Run : {RUN_ID}")
print(f"Dataset Version: {DATASET_VERSION}")
print(f"Pipeline       : {latest_run.pipeline_name}")
print(f"Version        : {latest_run.pipeline_version}")


check_eq(
    "RUN",
    "Execution Status",
    latest_run.execution_status,
    "SUCCESS"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 6. Bronze
# MAGIC
# MAGIC Se verifica que las cinco entidades Bronze existan y contengan registros.
# MAGIC
# MAGIC Esto valida materialización de ingesta; la calidad semántica se valida
# MAGIC en el Quality Gate Bronze.

# COMMAND ----------

# ============================================================
# 6. BRONZE
# ============================================================

bronze_validation = []


for table_name in BRONZE_TABLES:

    row_count = (
        spark.table(
            table_name
        )
        .count()
    )

    bronze_validation.append(
        (
            table_name,
            row_count
        )
    )

    check_true(
        "BRONZE",
        f"{table_name} no vacía",
        row_count > 0,
        expected="rows > 0"
    )


bronze_validation_df = spark.createDataFrame(
    bronze_validation,
    [
        "table_name",
        "rows"
    ]
)


display(
    bronze_validation_df
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 7. Data Quality — Bronze y Silver
# MAGIC
# MAGIC Para cada capa se valida la última ejecución DQ disponible:
# MAGIC
# MAGIC 1. existe una ejecución;
# MAGIC 2. se ejecutaron todas las reglas catalogadas para la capa;
# MAGIC 3. no existen reglas `alta` con `paso = false`.
# MAGIC
# MAGIC Las reglas de criticidad media o baja permanecen visibles como
# MAGIC observaciones y no bloquean el Acceptance Gate.

# COMMAND ----------

# ============================================================
# 7. DATA QUALITY
# ============================================================

def validate_dq_layer(
    layer
):

    stage = f"DQ_{layer.upper()}"

    rules_df = (
        spark.table(
            DQ_RULE_TABLE
        )
        .filter(
            F.lower(
                F.col("capa")
            ) == layer.lower()
        )
    )


    catalog_rule_count = (
        rules_df
        .select(
            "regla_id"
        )
        .distinct()
        .count()
    )


    latest_execution = (
        spark.table(
            DQ_RESULT_TABLE
        )
        .filter(
            F.lower(
                F.col("capa")
            ) == layer.lower()
        )
        .orderBy(
            F.desc(
                "ejecucion_ts"
            )
        )
        .select(
            "ejecucion_id",
            "ejecucion_ts"
        )
        .first()
    )


    check_true(
        stage,
        "Existe ejecución DQ",
        latest_execution is not None
    )


    if latest_execution is None:
        return None


    execution_id = latest_execution.ejecucion_id


    results_df = (
        spark.table(
            DQ_RESULT_TABLE
        )
        .filter(
            F.col(
                "ejecucion_id"
            ) == execution_id
        )
        .filter(
            F.lower(
                F.col("capa")
            ) == layer.lower()
        )
    )


    result_rule_count = (
        results_df
        .select(
            "regla_id"
        )
        .distinct()
        .count()
    )


    duplicate_rule_results = (
        results_df
        .groupBy(
            "regla_id"
        )
        .count()
        .filter(
            F.col("count") != 1
        )
        .count()
    )


    critical_failures = (
        results_df
        .filter(
            F.lower(
                F.col(
                    "criticidad"
                )
            ) == "alta"
        )
        .filter(
            F.col(
                "paso"
            ) == False
        )
        .count()
    )


    check_true(
        stage,
        "Catálogo DQ contiene reglas",
        catalog_rule_count > 0,
        expected="rules > 0"
    )


    check_eq(
        stage,
        "Reglas ejecutadas vs catálogo",
        result_rule_count,
        catalog_rule_count
    )


    check_eq(
        stage,
        "Resultados duplicados por regla",
        duplicate_rule_results,
        0
    )


    check_eq(
        stage,
        "Fallas críticas",
        critical_failures,
        0
    )


    print(
        f"  Execution ID {layer}: "
        f"{execution_id}"
    )


    return execution_id


DQ_BRONZE_EXECUTION_ID = validate_dq_layer(
    "bronze"
)

DQ_SILVER_EXECUTION_ID = validate_dq_layer(
    "silver"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 8. Silver y Feature Engineering
# MAGIC
# MAGIC Se valida:
# MAGIC
# MAGIC - grano único de `slv_review`;
# MAGIC - grano único de `slv_review_features`;
# MAGIC - consistencia de la versión Silver utilizada por el Run cuando la
# MAGIC   columna `_silver_execution_id` está disponible.
# MAGIC
# MAGIC La comparación crítica de Trust & Safety **no utiliza el total actual
# MAGIC de Silver como referencia del Run**. El Run se valida contra su propia
# MAGIC evidencia de evaluación.
# MAGIC
# MAGIC > ⚠️ `slv_review_features` se reconstruye con `overwrite`. Por ello,
# MAGIC > `_silver_execution_id` describe el snapshot **actual** de Features y puede
# MAGIC > diferir del `dataset_version` de un Run histórico válido. Mientras
# MAGIC > `evaluation_run` no persista también `feature_execution_id`, esta
# MAGIC > comprobación se registra como `WARNING` y no bloquea el Acceptance Gate.

# COMMAND ----------

# ============================================================
# 8. SILVER REVIEW
# ============================================================

silver_review = spark.table(
    REVIEW_TABLE
)


silver_review_count = (
    silver_review
    .count()
)


silver_review_distinct = (
    silver_review
    .select(
        "review_id"
    )
    .distinct()
    .count()
)


check_eq(
    "SILVER",
    "slv_review - grano review_id",
    silver_review_distinct,
    silver_review_count
)


# ============================================================
# 8.1 FEATURE ENGINEERING
# ============================================================

features = spark.table(
    FEATURE_TABLE
)


feature_stats = (
    features
    .agg(
        F.count("*").alias(
            "rows"
        ),
        F.countDistinct(
            "review_id"
        ).alias(
            "reviews"
        )
    )
    .first()
)


check_eq(
    "FEATURES",
    "Feature review_id únicos",
    feature_stats["reviews"],
    feature_stats["rows"]
)


feature_columns = {
    field.name
    for field in features.schema.fields
}


if "_silver_execution_id" in feature_columns:

    feature_dataset_versions = (
        features
        .select(
            "_silver_execution_id"
        )
        .where(
            F.col(
                "_silver_execution_id"
            ).isNotNull()
        )
        .distinct()
    )


    feature_dataset_version_count = (
        feature_dataset_versions
        .count()
    )


    feature_matches_run = (
        feature_dataset_versions
        .filter(
            F.col(
                "_silver_execution_id"
            ) == DATASET_VERSION
        )
        .count()
    )


    check_eq(
        "FEATURES",
        "Cantidad de dataset_version en features",
        feature_dataset_version_count,
        1
    )


    check_eq(
        "FEATURES",
        "Features corresponden al dataset_version del Run",
        feature_matches_run,
        1,
        severity="WARNING"
    )

else:

    register_check(
        "FEATURES",
        "Trazabilidad _silver_execution_id",
        "columna ausente",
        "columna disponible",
        False,
        severity="WARNING"
    )


# COMMAND ----------

# MAGIC %md
# MAGIC # 9. Trust & Safety Evaluation
# MAGIC
# MAGIC Esta es la corrección más importante respecto al notebook anterior.
# MAGIC
# MAGIC Las reglas esperadas se obtienen de las **reglas realmente registradas
# MAGIC en el `Evaluation Run`**.
# MAGIC
# MAGIC No se compara contra las reglas `ACTIVE` actuales, porque el catálogo
# MAGIC puede cambiar después de una ejecución histórica.

# COMMAND ----------

# ============================================================
# 9. TRUST & SAFETY EVALUATION
# ============================================================

run_evaluations = (
    spark.table(
        EVALUATION_TABLE
    )
    .filter(
        F.col(
            "evaluation_run_id"
        ) == RUN_ID
    )
)


evaluation_stats = (
    run_evaluations
    .agg(
        F.count("*").alias(
            "rows"
        ),
        F.countDistinct(
            "review_id"
        ).alias(
            "reviews"
        ),
        F.countDistinct(
            "rule_version_id"
        ).alias(
            "rules"
        )
    )
    .first()
)


run_review_count = evaluation_stats["reviews"]
run_rule_count = evaluation_stats["rules"]
evaluation_row_count = evaluation_stats["rows"]


expected_evaluations = (
    run_review_count
    *
    run_rule_count
)


distinct_evaluation_grain = (
    run_evaluations
    .select(
        "review_id",
        "rule_version_id",
        "evaluation_run_id"
    )
    .distinct()
    .count()
)


check_true(
    "EVALUATION",
    "Reviews evaluadas",
    run_review_count > 0,
    expected="reviews > 0"
)


check_true(
    "EVALUATION",
    "Reglas utilizadas por el Run",
    run_rule_count > 0,
    expected="rules > 0"
)


check_eq(
    "EVALUATION",
    "Review × Rule Version × Run",
    evaluation_row_count,
    expected_evaluations
)


check_eq(
    "EVALUATION",
    "Grano único Evaluation",
    distinct_evaluation_grain,
    evaluation_row_count
)


# ------------------------------------------------------------
# Validar que las versiones usadas por el Run existan
# en el catálogo histórico, sin exigir que sigan ACTIVE.
# ------------------------------------------------------------

run_rule_versions = (
    run_evaluations
    .select(
        "rule_version_id"
    )
    .distinct()
)


catalog_rule_versions = (
    spark.table(
        RULE_VERSION_TABLE
    )
    .select(
        "rule_version_id"
    )
    .distinct()
)


missing_rule_versions = (
    run_rule_versions
    .join(
        catalog_rule_versions,
        "rule_version_id",
        "left_anti"
    )
    .count()
)


check_eq(
    "EVALUATION",
    "Versiones del Run ausentes en catálogo",
    missing_rule_versions,
    0
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 10. Risk Consolidation
# MAGIC
# MAGIC Para el `RUN_ID` seleccionado debe existir exactamente:
# MAGIC
# MAGIC **1 Review × 1 Evaluation Run**
# MAGIC
# MAGIC y cada Review debe consolidar el mismo número de reglas que fueron
# MAGIC utilizadas en Evaluation.

# COMMAND ----------

# ============================================================
# 10. RISK CONSOLIDATION
# ============================================================

run_risk = (
    spark.table(
        RISK_TABLE
    )
    .filter(
        F.col(
            "evaluation_run_id"
        ) == RUN_ID
    )
)


risk_stats = (
    run_risk
    .agg(
        F.count("*").alias(
            "rows"
        ),
        F.countDistinct(
            "review_id"
        ).alias(
            "reviews"
        ),
        F.sum(
            F.when(
                F.col(
                    "requires_investigation"
                ),
                1
            )
            .otherwise(
                0
            )
        ).alias(
            "investigation"
        )
    )
    .first()
)


risk_row_count = risk_stats["rows"]
risk_review_count = risk_stats["reviews"]
risk_investigation_count = risk_stats["investigation"]


check_eq(
    "RISK",
    "Risk rows vs Reviews evaluadas",
    risk_row_count,
    run_review_count
)


check_eq(
    "RISK",
    "Risk review_id únicos",
    risk_review_count,
    risk_row_count
)


risk_rule_mismatch = (
    run_risk
    .filter(
        F.col(
            "rule_count"
        ) != run_rule_count
    )
    .count()
)


check_eq(
    "RISK",
    "Reviews con rule_count inconsistente",
    risk_rule_mismatch,
    0
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 11. Gold — trazabilidad del Run
# MAGIC
# MAGIC Gold se publica mediante snapshot del Run actual.
# MAGIC
# MAGIC Por eso no basta con comparar conteos: también se verifica que los
# MAGIC registros publicados correspondan exactamente al `RUN_ID`.

# COMMAND ----------

# ============================================================
# 11. GOLD REVIEW RISK
# ============================================================

gold_risk = spark.table(
    GOLD_REVIEW_RISK
)


gold_risk_stats = (
    gold_risk
    .agg(
        F.count("*").alias(
            "rows"
        ),
        F.countDistinct(
            "review_id"
        ).alias(
            "reviews"
        ),
        F.countDistinct(
            "evaluation_run_id"
        ).alias(
            "runs"
        )
    )
    .first()
)


gold_risk_run_rows = (
    gold_risk
    .filter(
        F.col(
            "evaluation_run_id"
        ) == RUN_ID
    )
    .count()
)


check_eq(
    "GOLD_RISK",
    "Filas Gold Risk",
    gold_risk_stats["rows"],
    risk_row_count
)


check_eq(
    "GOLD_RISK",
    "Review IDs únicos",
    gold_risk_stats["reviews"],
    risk_review_count
)


check_eq(
    "GOLD_RISK",
    "Evaluation Runs presentes",
    gold_risk_stats["runs"],
    1
)


check_eq(
    "GOLD_RISK",
    "Filas correspondientes al RUN_ID",
    gold_risk_run_rows,
    gold_risk_stats["rows"]
)


# ============================================================
# 11.1 GOLD FACT
# ============================================================

gold_fact = spark.table(
    GOLD_FACT_EVALUATION
)


gold_fact_stats = (
    gold_fact
    .agg(
        F.count("*").alias(
            "rows"
        ),
        F.countDistinct(
            "evaluation_run_id"
        ).alias(
            "runs"
        )
    )
    .first()
)


gold_fact_run_rows = (
    gold_fact
    .filter(
        F.col(
            "evaluation_run_id"
        ) == RUN_ID
    )
    .count()
)


gold_fact_distinct_grain = (
    gold_fact
    .select(
        "review_id",
        "rule_version_id",
        "evaluation_run_id"
    )
    .distinct()
    .count()
)


check_eq(
    "GOLD_FACT",
    "Filas Fact vs Evaluation",
    gold_fact_stats["rows"],
    evaluation_row_count
)


check_eq(
    "GOLD_FACT",
    "Filas Fact del RUN_ID",
    gold_fact_run_rows,
    gold_fact_stats["rows"]
)


check_eq(
    "GOLD_FACT",
    "Grano único Review × Rule × Run",
    gold_fact_distinct_grain,
    gold_fact_stats["rows"]
)


# ============================================================
# 11.2 INVESTIGATION QUEUE
# ============================================================

gold_queue = spark.table(
    GOLD_QUEUE
)


queue_count = (
    gold_queue
    .count()
)


queue_distinct_reviews = (
    gold_queue
    .select(
        "review_id"
    )
    .distinct()
    .count()
)


expected_queue_ids = (
    run_risk
    .filter(
        F.col(
            "requires_investigation"
        ) == True
    )
    .select(
        "review_id"
    )
    .distinct()
)


actual_queue_ids = (
    gold_queue
    .select(
        "review_id"
    )
    .distinct()
)


missing_queue_reviews = (
    expected_queue_ids
    .join(
        actual_queue_ids,
        "review_id",
        "left_anti"
    )
    .count()
)


unexpected_queue_reviews = (
    actual_queue_ids
    .join(
        expected_queue_ids,
        "review_id",
        "left_anti"
    )
    .count()
)


check_eq(
    "GOLD_QUEUE",
    "Cantidad de candidatos",
    queue_count,
    risk_investigation_count
)


check_eq(
    "GOLD_QUEUE",
    "Review IDs únicos",
    queue_distinct_reviews,
    queue_count
)


check_eq(
    "GOLD_QUEUE",
    "Candidatos faltantes",
    missing_queue_reviews,
    0
)


check_eq(
    "GOLD_QUEUE",
    "Candidatos inesperados",
    unexpected_queue_reviews,
    0
)


# ============================================================
# 11.3 KPI OVERVIEW
# ============================================================

overview_df = spark.table(
    GOLD_OVERVIEW
)


overview_count = (
    overview_df
    .count()
)


check_eq(
    "GOLD_KPI",
    "Overview - filas",
    overview_count,
    1
)


overview = (
    overview_df
    .first()
)


if overview is not None:

    check_eq(
        "GOLD_KPI",
        "Overview - Evaluation Run",
        overview["evaluation_run_id"],
        RUN_ID
    )


    check_eq(
        "GOLD_KPI",
        "Overview - Reviews",
        overview["reviews_evaluated"],
        risk_row_count
    )


    check_eq(
        "GOLD_KPI",
        "Overview - Investigación",
        overview["reviews_for_investigation"],
        queue_count
    )


# COMMAND ----------

# MAGIC %md
# MAGIC # 12. Gold — cobertura de productos
# MAGIC
# MAGIC Se comprueba que todos los productos definidos por `06_gold_productos`
# MAGIC estén publicados y contengan registros.
# MAGIC
# MAGIC Para dimensiones/KPIs, `rows > 0` es una validación estructural mínima;
# MAGIC las reconciliaciones críticas se realizan sobre Risk, Fact, Queue y Overview.

# COMMAND ----------

# ============================================================
# 12. COBERTURA GOLD
# ============================================================

gold_product_rows = []


for table_name in GOLD_TABLES:

    full_name = f"{GOLD}.{table_name}"

    row_count = (
        spark.table(
            full_name
        )
        .count()
    )

    gold_product_rows.append(
        (
            table_name,
            row_count
        )
    )

    check_true(
        "GOLD_PRODUCTS",
        f"{table_name} no vacío",
        row_count > 0,
        expected="rows > 0"
    )


gold_product_rows_df = spark.createDataFrame(
    gold_product_rows,
    [
        "gold_product",
        "rows"
    ]
)


display(
    gold_product_rows_df
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 13. Prioridades
# MAGIC
# MAGIC Toda Review Gold debe pertenecer exactamente a una categoría de
# MAGIC `investigation_priority`.
# MAGIC
# MAGIC La suma de prioridades debe reconciliar con `gld_review_risk`.

# COMMAND ----------

# ============================================================
# 13. PRIORIDADES
# ============================================================

priority_stats = (
    gold_risk
    .groupBy(
        "investigation_priority"
    )
    .count()
)


display(
    priority_stats
)


priority_total = (
    priority_stats
    .agg(
        F.sum(
            "count"
        ).alias(
            "total"
        )
    )
    .first()
    .total
)


null_priority_count = (
    gold_risk
    .filter(
        F.col(
            "investigation_priority"
        ).isNull()
    )
    .count()
)


check_eq(
    "PRIORITY",
    "Total prioridades",
    priority_total,
    risk_row_count
)


check_eq(
    "PRIORITY",
    "Prioridades NULL",
    null_priority_count,
    0
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 14. Unity Catalog — metadata de productos Gold
# MAGIC
# MAGIC El Acceptance Gate verifica que los 11 productos Gold tengan, como mínimo:
# MAGIC
# MAGIC - `data_domain`;
# MAGIC - `data_layer`;
# MAGIC - `data_product`;
# MAGIC - `technical_owner`;
# MAGIC - `business_owner`;
# MAGIC - `data_steward`.
# MAGIC
# MAGIC Esto evita publicar tablas técnicamente correctas pero sin gobierno
# MAGIC mínimo en Unity Catalog.

# COMMAND ----------

# ============================================================
# 14. METADATA GOLD
# ============================================================

required_table_tags = {
    "data_domain",
    "data_layer",
    "data_product",
    "technical_owner",
    "business_owner",
    "data_steward"
}


table_tags_df = spark.sql(
    f"""
    SELECT
        table_name,
        tag_name,
        tag_value
    FROM {CATALOG}.information_schema.table_tags
    WHERE schema_name = '{GOLD_SCHEMA}'
      AND table_name LIKE 'gld_%'
    """
)


gold_table_names = set(
    GOLD_TABLES
)


observed_tags = {
    (
        row.table_name,
        row.tag_name
    )
    for row in (
        table_tags_df
        .filter(
            F.col(
                "table_name"
            ).isin(
                GOLD_TABLES
            )
        )
        .select(
            "table_name",
            "tag_name"
        )
        .collect()
    )
}


expected_tag_pairs = {
    (
        table_name,
        tag_name
    )
    for table_name in GOLD_TABLES
    for tag_name in required_table_tags
}


missing_tag_pairs = (
    expected_tag_pairs
    -
    observed_tags
)


check_eq(
    "UNITY_CATALOG",
    "Tags mínimos faltantes en Gold",
    len(missing_tag_pairs),
    0
)


if missing_tag_pairs:

    print()
    print("Metadata Gold faltante:")

    for table_name, tag_name in sorted(
        missing_tag_pairs
    ):
        print(
            f" - {table_name}: {tag_name}"
        )


display(
    table_tags_df
    .filter(
        F.col(
            "table_name"
        ).isin(
            GOLD_TABLES
        )
    )
    .orderBy(
        "table_name",
        "tag_name"
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 15. Clasificación de columnas sensibles
# MAGIC
# MAGIC Se valida la clasificación definida por el control de gobierno `E6.3`.
# MAGIC
# MAGIC ### Columnas esperadas
# MAGIC
# MAGIC **Silver Review**
# MAGIC
# MAGIC - `user_id → restricted`
# MAGIC - `review_text → confidential`
# MAGIC
# MAGIC **Gold Review Risk / Investigation Queue**
# MAGIC
# MAGIC - identificadores de usuario → `restricted`;
# MAGIC - texto y señales de riesgo → `confidential`.
# MAGIC
# MAGIC Si `STRICT_GOVERNANCE = True`, una clasificación faltante bloquea
# MAGIC el Acceptance Gate.

# COMMAND ----------

# ============================================================
# 15. CLASIFICACIÓN DE COLUMNAS
# ============================================================

required_classifications = {

    (
        SILVER_SCHEMA,
        "slv_review",
        "user_id"
    ):
        "restricted",

    (
        SILVER_SCHEMA,
        "slv_review",
        "review_text"
    ):
        "confidential",

    (
        GOLD_SCHEMA,
        "gld_investigation_queue",
        "user_id"
    ):
        "restricted",

    (
        GOLD_SCHEMA,
        "gld_investigation_queue",
        "user_sk"
    ):
        "restricted",

    (
        GOLD_SCHEMA,
        "gld_investigation_queue",
        "review_text"
    ):
        "confidential",

    (
        GOLD_SCHEMA,
        "gld_investigation_queue",
        "weighted_risk_score"
    ):
        "confidential",

    (
        GOLD_SCHEMA,
        "gld_investigation_queue",
        "risk_classification"
    ):
        "confidential",

    (
        GOLD_SCHEMA,
        "gld_investigation_queue",
        "investigation_priority"
    ):
        "confidential",

    (
        GOLD_SCHEMA,
        "gld_review_risk",
        "user_id"
    ):
        "restricted",

    (
        GOLD_SCHEMA,
        "gld_review_risk",
        "review_text"
    ):
        "confidential",

    (
        GOLD_SCHEMA,
        "gld_review_risk",
        "weighted_risk_score"
    ):
        "confidential",

    (
        GOLD_SCHEMA,
        "gld_review_risk",
        "risk_classification"
    ):
        "confidential",

    (
        GOLD_SCHEMA,
        "gld_review_risk",
        "investigation_priority"
    ):
        "confidential"
}


column_tags_df = spark.sql(
    f"""
    SELECT
        schema_name,
        table_name,
        column_name,
        tag_name,
        tag_value
    FROM {CATALOG}.information_schema.column_tags
    WHERE schema_name IN (
        '{SILVER_SCHEMA}',
        '{GOLD_SCHEMA}'
    )
      AND tag_name = 'classification'
    """
)


observed_classifications = {
    (
        row.schema_name,
        row.table_name,
        row.column_name
    ):
        row.tag_value

    for row in (
        column_tags_df
        .collect()
    )
}


classification_issues = []


for key, expected_value in required_classifications.items():

    actual_value = observed_classifications.get(
        key
    )

    if actual_value != expected_value:

        classification_issues.append(
            (
                key,
                expected_value,
                actual_value
            )
        )


classification_severity = (
    "CRITICAL"
    if STRICT_GOVERNANCE
    else "WARNING"
)


register_check(
    "CLASSIFICATION",
    "Clasificaciones faltantes o incorrectas",
    len(classification_issues),
    0,
    len(classification_issues) == 0,
    severity=classification_severity
)


if classification_issues:

    print()
    print("Clasificaciones pendientes:")

    for (
        schema_name,
        table_name,
        column_name
    ), expected_value, actual_value in classification_issues:

        print(
            f" - {schema_name}.{table_name}.{column_name}: "
            f"actual={actual_value}, "
            f"esperado={expected_value}"
        )


display(
    column_tags_df
    .orderBy(
        "schema_name",
        "table_name",
        "column_name"
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 16. Resultado consolidado
# MAGIC
# MAGIC El scorecard final permite identificar exactamente qué etapa bloqueó
# MAGIC la publicación.
# MAGIC
# MAGIC El Acceptance Gate falla únicamente por checks `CRITICAL`.

# COMMAND ----------

# ============================================================
# 16. RESULTADOS
# ============================================================

validation_df = spark.createDataFrame(
    checks,
    [
        "stage",
        "validation",
        "actual",
        "expected",
        "severity",
        "passed"
    ]
)


display(
    validation_df
    .orderBy(
        "stage",
        "validation"
    )
)


# COMMAND ----------

# ============================================================
# 17. SCORECARD POR ETAPA
# ============================================================

stage_scorecard = (
    validation_df

    .groupBy(
        "stage"
    )

    .agg(
        F.count("*").alias(
            "checks"
        ),

        F.sum(
            F.when(
                F.col(
                    "passed"
                ) == True,
                1
            )
            .otherwise(
                0
            )
        ).alias(
            "passed"
        ),

        F.sum(
            F.when(
                (
                    F.col(
                        "passed"
                    ) == False
                )
                &
                (
                    F.col(
                        "severity"
                    ) == "CRITICAL"
                ),
                1
            )
            .otherwise(
                0
            )
        ).alias(
            "critical_failed"
        ),

        F.sum(
            F.when(
                (
                    F.col(
                        "passed"
                    ) == False
                )
                &
                (
                    F.col(
                        "severity"
                    ) == "WARNING"
                ),
                1
            )
            .otherwise(
                0
            )
        ).alias(
            "warnings"
        )
    )

    .orderBy(
        "stage"
    )
)


display(
    stage_scorecard
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 18. Acceptance Gate
# MAGIC
# MAGIC Criterio:
# MAGIC
# MAGIC ```text
# MAGIC critical_failed = 0
# MAGIC          ↓
# MAGIC PIPELINE APROBADO
# MAGIC ```
# MAGIC
# MAGIC Cualquier check crítico fallido evita declarar la capa Gold lista
# MAGIC para consumo.

# COMMAND ----------

# ============================================================
# 18. ACCEPTANCE GATE
# ============================================================

failed_critical = (
    validation_df

    .filter(
        F.col(
            "passed"
        ) == False
    )

    .filter(
        F.col(
            "severity"
        ) == "CRITICAL"
    )

    .count()
)


warning_count = (
    validation_df

    .filter(
        F.col(
            "passed"
        ) == False
    )

    .filter(
        F.col(
            "severity"
        ) == "WARNING"
    )

    .count()
)


print()
print("=" * 78)
print("RESULTADO ACCEPTANCE GATE END-TO-END")
print("=" * 78)

print(
    f"Evaluation Run       : {RUN_ID}"
)

print(
    f"Dataset Version      : {DATASET_VERSION}"
)

print(
    f"Checks ejecutados    : {len(checks)}"
)

print(
    f"Fallos críticos      : {failed_critical}"
)

print(
    f"Warnings             : {warning_count}"
)

print("=" * 78)


if failed_critical > 0:

    print()
    print(
        "❌ PIPELINE END-TO-END RECHAZADO"
    )

    display(
        validation_df

        .filter(
            F.col(
                "passed"
            ) == False
        )

        .filter(
            F.col(
                "severity"
            ) == "CRITICAL"
        )
    )

    raise Exception(
        f"""
        ❌ ACCEPTANCE GATE FALLÓ.

        Fallos críticos:
        {failed_critical}

        Evaluation Run:
        {RUN_ID}
        """
    )


print()
print(
    "✔ PIPELINE END-TO-END APROBADO"
)

print(
    "✔ Evaluation, Risk y Gold corresponden al mismo Run."
)

print(
    "✔ Quality Gates Bronze y Silver superados."
)

print(
    "✔ Metadata mínima Gold validada."
)

if warning_count > 0:

    print(
        f"⚠ Existen {warning_count} warnings no bloqueantes."
    )


print()
print(
    "✔ La capa Gold está lista para consumo Power BI."
)

print("=" * 78)


# COMMAND ----------

# MAGIC %md
# MAGIC # ✅ Definition of Done — End-to-End
# MAGIC
# MAGIC El Acceptance Gate se considera completo cuando:
# MAGIC
# MAGIC - [ ] Las cinco tablas Bronze existen y contienen datos.
# MAGIC - [ ] La última ejecución DQ Bronze cubre todas las reglas catalogadas.
# MAGIC - [ ] DQ Bronze no tiene fallas de criticidad alta.
# MAGIC - [ ] `slv_review` mantiene grano único por `review_id`.
# MAGIC - [ ] La última ejecución DQ Silver cubre todas las reglas catalogadas.
# MAGIC - [ ] DQ Silver no tiene fallas de criticidad alta.
# MAGIC - [ ] Features mantiene grano único por `review_id`.
# MAGIC - [ ] El `dataset_version` del Run es trazable desde Features cuando existe metadata.
# MAGIC - [ ] Evaluation utiliza el grano `Review × Rule Version × Run`.
# MAGIC - [ ] Las reglas del Run se validan contra su evidencia histórica, no contra ACTIVE actual.
# MAGIC - [ ] Todas las `rule_version_id` utilizadas existen en el catálogo.
# MAGIC - [ ] Risk mantiene `1 Review × 1 Evaluation Run`.
# MAGIC - [ ] `rule_count` de Risk coincide con las reglas del Run.
# MAGIC - [ ] Gold Review Risk corresponde exclusivamente al `RUN_ID`.
# MAGIC - [ ] Gold Fact corresponde exclusivamente al `RUN_ID`.
# MAGIC - [ ] Investigation Queue coincide exactamente con `requires_investigation = true`.
# MAGIC - [ ] KPI Overview corresponde al mismo `RUN_ID`.
# MAGIC - [ ] Los 11 productos Gold están publicados.
# MAGIC - [ ] Las prioridades reconcilian con el total de Reviews.
# MAGIC - [ ] Todos los productos Gold poseen metadata mínima en Unity Catalog.
# MAGIC - [ ] Las columnas sensibles poseen clasificación según E6.3.
# MAGIC - [ ] Ningún resultado heurístico se interpreta como fraude confirmado.
